# Data Cleaning and Preparation

Many researchers choose to do ``ad hoc`` processing of data from one form to another using a general-purpose programming language, like Python, Perl, R, or Java, or Unix text-processing tools like sed or awk. 

Here we discuss tools for missing data, duplicate data, string manipulation, and some other analytical data transformations.


## Handling Missing Data

Missing data occurs commonly in many data analysis applications. One of the goals of pandas is to make working with missing data as painless as possible. For example, all of the descriptive statistics on pandas objects exlude missing data by default. 

The way that missing data is represented in pandas objects is somewhat imperfect, but it is sufficient for most real-world use. For data with ``float64`` dtype, pandas uses the floating-point value ``NaN`` to represent missing data. 


We call this a *sentinel value*: when present, it indicates a missing (or *null*) value:

In [2]:
import numpy as np 
import pandas as pd 

float_data = pd.Series([1.2, -3.5, np.nan,0])

float_data

0    1.2
1   -3.5
2    NaN
3    0.0
dtype: float64

The ``isna`` method gives us a Boolean Series with ``True`` where values are null:

In [3]:
float_data.isna()

0    False
1    False
2     True
3    False
dtype: bool

When cleaning up data for analysis, it is often important to do analysis on the missing data itself to identify data collection problems biases in the data caused by missing data. 

The built-in Python ``None`` value is also treated as NA:

In [4]:
string_data = pd.Series(["aardvark", np.nan, None, "avocado"])

string_data

0    aardvark
1         NaN
2        None
3     avocado
dtype: object

In [5]:
string_data.isna()

0    False
1     True
2     True
3    False
dtype: bool

In [6]:
float_data = pd.Series([1, 2, None], dtype = 'float64')

float_data

0    1.0
1    2.0
2    NaN
dtype: float64

In [7]:
float_data.isna()

0    False
1    False
2     True
dtype: bool

List of some functions related to missing data handling:

| Method | Description |
|---|---|
| `dropna` | Filter axis labels based on whether values for each label have missing data, with varying thresholds for how much missing data to tolerate. |
|``fillna``|Fill missing data with some value or using an interpolation method such as ``"ffill"`` or ``"bfill"``|
|``isna``|Return Boolen values indicating which values are missing/NA.|
|``notna``|Negation of ``isna``, returns ``True`` for non-NA values and ``False`` for NA values|


## Filtering Out Missing Data

There are a few ways to filter out missing data. While you always have the option to do it by hand using ``pandas.isna`` and Boolean indexing, ``dropna`` can be helpful. On a Series, it returns the Series with only the nonnull data and index values:

In [8]:
data = pd.Series([1, np.nan, 3.5, np.nan, 7])

data.dropna()

0    1.0
2    3.5
4    7.0
dtype: float64

This is the same thing as doing:

In [9]:
data[data.notna()]

0    1.0
2    3.5
4    7.0
dtype: float64

With DataFrame objects, there are different ways to remove missing data. You may want to drop rows or columns that are all NA, or only those rows or columns containing any NAs at all. ``dropna`` by default drops any row containing a missing value:

In [10]:
data = pd.DataFrame([[1., 6.5, 3., ], [1., np.nan, np.nan], [np.nan, np.nan, np.nan], [np.nan, 6.5, 3.]])

data

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
2,NaN,NaN,NaN
3,NaN,6.5,3.0


In [11]:
data.dropna()

,0,1,2
0,1.0,6.5,3.0


Passing ``how="all"`` will drop only rows that are all NA:

In [12]:
data.dropna(how="all")

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
3,NaN,6.5,3.0


To drop columns in the same way, pass  ``axis = "columns"``:

In [13]:
data[4] = np.nan

data

,0,1,2,4
0,1.0,6.5,3.0,NaN
1,1.0,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,6.5,3.0,NaN


In [14]:
data.dropna(axis="columns", how="all")

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
2,NaN,NaN,NaN
3,NaN,6.5,3.0


Suppose you want to keep only rows containing at most a certain number of missing observations. You can indicate this with the ``thresh`` argument:

In [15]:
df = pd.DataFrame(np.random.standard_normal((7, 3)))

df.iloc[:4, 1] = np.nan

df.iloc[:2, 2] = np.nan

df

,0,1,2
0,0.314185,NaN,NaN
1,-2.382690,NaN,NaN
2,0.086637,NaN,0.550146
3,0.545553,NaN,-1.732867
4,-1.680972,1.665200,0.564489
5,-0.759086,-2.953491,-1.558721
6,-0.961597,0.007814,-1.071041


In [16]:
df.dropna()

,0,1,2
4,-1.680972,1.665200,0.564489
5,-0.759086,-2.953491,-1.558721
6,-0.961597,0.007814,-1.071041


In [17]:
df.dropna(thresh=2)

,0,1,2
2,0.086637,NaN,0.550146
3,0.545553,NaN,-1.732867
4,-1.680972,1.665200,0.564489
5,-0.759086,-2.953491,-1.558721
6,-0.961597,0.007814,-1.071041


## Filling in Missing Data 

Rather than filtering out missing data (and potentially discarding other data along with it), you may want to fill in the "holes" in any number of ways. For most purposes, the ``fillna`` method is the workhorse function to use. Calling ``fillna`` with a constant replaces missing values with that value:

In [18]:
df.fillna(0)

,0,1,2
0,0.314185,0.000000,0.000000
1,-2.382690,0.000000,0.000000
2,0.086637,0.000000,0.550146
3,0.545553,0.000000,-1.732867
4,-1.680972,1.665200,0.564489
5,-0.759086,-2.953491,-1.558721
6,-0.961597,0.007814,-1.071041


Calling ``fillna`` with a dictionary, you can use a different full value for each column:

In [19]:
df.fillna({1: 0.5, 2:0})

,0,1,2
0,0.314185,0.500000,0.000000
1,-2.382690,0.500000,0.000000
2,0.086637,0.500000,0.550146
3,0.545553,0.500000,-1.732867
4,-1.680972,1.665200,0.564489
5,-0.759086,-2.953491,-1.558721
6,-0.961597,0.007814,-1.071041


The same interpolation methods available for reindexing can be used with ``fillna``:


In [20]:
df = pd.DataFrame(np.random.standard_normal((6, 3)))

df.iloc[2:, 1] = np.nan

df.iloc[4:, 2] = np.nan

df

,0,1,2
0,-1.368012,0.729207,0.119796
1,-0.771573,1.051605,-0.217246
2,0.935513,NaN,0.881459
3,-0.733532,NaN,-0.481039
4,-0.717154,NaN,NaN
5,0.163502,NaN,NaN


In [21]:
df.ffill()

,0,1,2
0,-1.368012,0.729207,0.119796
1,-0.771573,1.051605,-0.217246
2,0.935513,1.051605,0.881459
3,-0.733532,1.051605,-0.481039
4,-0.717154,1.051605,-0.481039
5,0.163502,1.051605,-0.481039


In [22]:
df.ffill(limit=2)

,0,1,2
0,-1.368012,0.729207,0.119796
1,-0.771573,1.051605,-0.217246
2,0.935513,1.051605,0.881459
3,-0.733532,1.051605,-0.481039
4,-0.717154,NaN,-0.481039
5,0.163502,NaN,-0.481039


With ``fillna`` you can do lots of other things such as simple data imputation using the median and mean statistics:

In [23]:
data = pd.Series([1., np.nan, 3.5, np.nan, 7])

data.fillna(data.mean())

0    1.000000
1    3.833333
2    3.500000
3    3.833333
4    7.000000
dtype: float64

# Data Transformation

Filtering, cleaning, and other transformations are another class of important operations. 

## Removing Duplicates 

Duplicate rows may be found in a DataFrame for any number of reasons. Here is an example:

In [24]:
data = pd.DataFrame({"k1": ["one", "two"] * 3 + ["two"],
                    "k2": [1, 1, 2, 3, 3, 4, 4]})
data

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4
6,two,4


The DataFrame method ``duplicated`` returns a Boolean Series indicating whether each row is a duplicate (its column values are exactly equal  to those in an earlier row) or not:

In [25]:
data.duplicated()

0    False
1    False
2    False
3    False
4    False
5    False
6     True
dtype: bool

Relatedly, ``drop_duplcates`` returns a DataFrame with rows where the ``duplicated`` array is ``False`` filtered out:

In [26]:
data.drop_duplicates()

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4


Both methods by default consider all of the columns; alternatively, you can specify any subset of them to detect duplicates. Suppose we had an additional column of values and wanted to filter duplicates based on the "k1" column:

In [27]:
data["v1"] = range(7)

data

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
5,two,4,5
6,two,4,6


In [28]:
data.drop_duplicates(subset=["k1"])

,k1,k2,v1
0,one,1,0
1,two,1,1


``duplicated`` and ``drop_duplicates`` by default keep the first observed value combination. Passing ``keep="last"`` will return the last one:

In [29]:
data.drop_duplicates(["k1", "k2"], keep="last")

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
6,two,4,6


## Transforming Data Using a Function or Mapping

For many datasets, you may wish to perform some transformation based on the values in the array, Series, or column in a DataFrame. Consider the following hypothetical data collected about various kinds of meat:

In [30]:
data = pd.DataFrame({"food": ["bacon", "pulled pork", "bacon",
                            "pastrami", "corned beef", "bacon",
                            "pastrami", "honey ham", "nova lox"],
                    "ounces": [4, 3, 12, 6, 7.5, 8, 3, 5, 6]})
data

,food,ounces
0,bacon,4.0
1,pulled pork,3.0
2,bacon,12.0
3,pastrami,6.0
4,corned beef,7.5
5,bacon,8.0
6,pastrami,3.0
7,honey ham,5.0
8,nova lox,6.0


Suppose you wanted to add a column indicating the type of animal that each food came from. Let's write down a mapping of each distinct meat type to the kind of animal:

In [31]:
meat_to_animal = {
"bacon": "pig",
"pulled pork": "pig",
"pastrami": "cow",
"corned beef": "cow",
"honey ham": "pig",
"nova lox": "salmon"
}

The ``map`` method on a Series accepts a function or dictionary-like object containing a mapping to do the transformation of values:

In [32]:
data["animal"] = data["food"].map(meat_to_animal)

data 

,food,ounces,animal
0,bacon,4.0,pig
1,pulled pork,3.0,pig
2,bacon,12.0,pig
3,pastrami,6.0,cow
4,corned beef,7.5,cow
5,bacon,8.0,pig
6,pastrami,3.0,cow
7,honey ham,5.0,pig
8,nova lox,6.0,salmon


We could also have passed a function that does all the work:

In [33]:
def get_animal(x):
    return meat_to_animal[x]

data["food"].map(get_animal)

0       pig
1       pig
2       pig
3       cow
4       cow
5       pig
6       cow
7       pig
8    salmon
Name: food, dtype: object

Using ``map`` is a convinient way to perform element-wise transformations and other data cleaning-relation operations.

## Replacing Values

Filling in missing data with the ``fillna`` method is a special case of more general value replacement. 

``map`` can be used to modify a subset of values in an object, but ``replace`` provides a simpler and more flexible way to do so. Let's consider this series:

In [34]:
data = pd.Series([1., -999., -1000., 3.])

data 

0       1.0
1    -999.0
2   -1000.0
3       3.0
dtype: float64

The ``-999`` values might be sentinel values for missing data. To replace these with NA values that pandas understands, we can use ``replace``, producing a new Series:

In [35]:
data.replace(-999, np.nan)



0       1.0
1       NaN
2   -1000.0
3       3.0
dtype: float64

If you want to replace multiple values at once, you instead pass a list and then the substitue value:

In [36]:
data.replace([-999, -1000], np.nan)

0    1.0
1    NaN
2    NaN
3    3.0
dtype: float64

To use a different replacement for each value, pass a list of substites:

In [37]:
data.replace([-999, -1000], [np.nan, 0])

0    1.0
1    NaN
2    0.0
3    3.0
dtype: float64

The argument passed can be a dictionary:

In [38]:
data.replace({-999: np.nan, -1000:0})

0    1.0
1    NaN
2    0.0
3    3.0
dtype: float64

## Renaming Axis Indexes

Like values in a Series, axis labels can be similarly transformed by a function or mapping of some form to produce new, differently labeled objects. You can alse modify the axes in place without creating a new data structure. Here's a simple example:

In [39]:
data = pd.DataFrame(np.arange(12).reshape((3, 4)), 
                    index=["Ohio", "Colorado", "New York"],
                    columns=["one", "two", "three", "four"])

data 

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
New York,8,9,10,11


Like a Series, the axis indexes have a ``map`` method:

In [40]:
def transform(x):
    return x[:4].upper()

data.index.map(transform)

Index(['OHIO', 'COLO', 'NEW '], dtype='object')

You can assign to the ``index`` attribute, modifying the DataFrame in place:

In [41]:
data.index = data.index.map(transform)

data 

,one,two,three,four
OHIO,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


If you want to create a transformed version of a dataset without modifying the original, a useful method is ``rename``:

In [42]:
data.rename(index=str.title, columns=str.upper)

,ONE,TWO,THREE,FOUR
Ohio,0,1,2,3
Colo,4,5,6,7
New,8,9,10,11


Notably, ``rename`` can be used in conjunction with a dictionary-like object, providing new values for a subset of the axis labels:

In [43]:
data.rename(index={"OHIO": "INDIANA"}, columns={"three": "peekaboo"})



,one,two,peekaboo,four
INDIANA,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


``rename`` saves you from the chore of copying the DataFrame manually and assigning new values to its ``index`` and ``columns`` attributes.


## Discretization and Binning 

Continous data is often discretized or otherwise separated into "bins" for analysis. Suppose you have data about a group of people in a study, and you want to group them into a discrete age buckets:

In [44]:
ages = [20, 22, 25, 27, 21, 23, 37, 31, 61, 45, 41, 32]

Let's divide these into bins of 18 to 25, 26 to 35, 36 to 60, and finally 61 and older. To do so, you have to use ``pandas.cut``:

In [45]:
bins = [18, 25, 35, 60, 100]

In [46]:
age_categories = pd.cut(ages, bins)

age_categories

[(18, 25], (18, 25], (18, 25], (25, 35], (18, 25], ..., (25, 35], (60, 100], (35, 60], (35, 60], (25, 35]]
Length: 12
Categories (4, interval[int64, right]): [(18, 25] < (25, 35] < (35, 60] < (60, 100]]

The object pandas returns is a special Categorical object. The output you see describes the bins computed by ``pandas.cut``. Each bin is identified by a special interval value type containing the lower and upper limit of each bin:

In [47]:
age_categories.codes

array([0, 0, 0, 1, 0, 0, 2, 1, 3, 2, 2, 1], dtype=int8)

In [48]:
age_categories.categories

IntervalIndex([(18, 25], (25, 35], (35, 60], (60, 100]], dtype='interval[int64, right]')

In [49]:
age_categories.categories[0]

Interval(18, 25, closed='right')

Note that ``pd.values_counts(categories)`` are the bin counts for the result of ``pandas.cut`` 

In [50]:
pd.cut(ages, bins, right=False)

[[18, 25), [18, 25), [25, 35), [25, 35), [18, 25), ..., [25, 35), [60, 100), [35, 60), [35, 60), [25, 35)]
Length: 12
Categories (4, interval[int64, left]): [[18, 25) < [25, 35) < [35, 60) < [60, 100)]

You can override the default interval-based bin labeling by passing a list or array to the ``labels`` option:

In [51]:
group_names = ["Youth", "YoungAdult", "MiddleAged", "Senior"]

pd.cut(ages, bins, labels=group_names)

['Youth', 'Youth', 'Youth', 'YoungAdult', 'Youth', ..., 'YoungAdult', 'Senior', 'MiddleAged', 'MiddleAged', 'YoungAdult']
Length: 12
Categories (4, object): ['Youth' < 'YoungAdult' < 'MiddleAged' < 'Senior']

If you pass an integer number of bins to ``pandas.cut`` instead of explicit bin edges, it will compute equal length bins based on the minimum and maximum values in the data. Consider the case of some uniformly distributed data chopped into fourths:

In [52]:
data = np.random.uniform(size = 20)

pd.cut(data, 4, precision=2)

[(0.75, 1.0], (0.0083, 0.26], (0.75, 1.0], (0.26, 0.5], (0.0083, 0.26], ..., (0.75, 1.0], (0.26, 0.5], (0.0083, 0.26], (0.75, 1.0], (0.75, 1.0]]
Length: 20
Categories (4, interval[float64, right]): [(0.0083, 0.26] < (0.26, 0.5] < (0.5, 0.75] < (0.75, 1.0]]

The ``precision=2`` option limits the decimal precision to two digits. 

A closely related function, ``pandas.qcut`` bins the data based on sample quantiles. Depending on the distribution of the data, using ``pandas.cut`` will not usually result in each bin having the same number of data points. Since ``pandas.qcut`` uses sample quantiles instead, you will obtain roughly equally sized bins:

In [54]:
data = np.random.standard_normal(1000)

quartiles = pd.qcut(data, 4, precision=2)

quartiles

[(-3.01, -0.69], (-3.01, -0.69], (-0.0021, 0.71], (-3.01, -0.69], (0.71, 3.24], ..., (0.71, 3.24], (0.71, 3.24], (0.71, 3.24], (-0.69, -0.0021], (0.71, 3.24]]
Length: 1000
Categories (4, interval[float64, right]): [(-3.01, -0.69] < (-0.69, -0.0021] < (-0.0021, 0.71] < (0.71, 3.24]]

In [56]:
pd.Series(quartiles).value_counts()

(-3.01, -0.69]      250
(-0.69, -0.0021]    250
(-0.0021, 0.71]     250
(0.71, 3.24]        250
Name: count, dtype: int64

# Detecting and Filtering Outliers 

Filtering or transforming outliers is largely a matter of applying array operations. Consider a DataFrame with some normally distributed data:

In [57]:
data = pd.DataFrame(np.random.standard_normal((1000,4)))

data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,0.000220,0.001730,-0.026181,-0.018126
std,1.019770,0.977340,0.993341,1.031745
min,-2.842948,-3.108153,-3.458135,-3.088709
25%,-0.748958,-0.652075,-0.674084,-0.712238
50%,0.026256,-0.053696,-0.050211,-0.000837
75%,0.673777,0.609422,0.651785,0.740628
max,3.318278,3.761675,3.594415,2.907953


Suppose you wanted to find values in one of the columns exceeding 3 in absolute value:

In [58]:
col = data[2]

col[col.abs() > 3]

188   -3.458135
433   -3.048507
639    3.387302
951    3.594415
Name: 2, dtype: float64

To select all rows having a value exceeding 3 or -3, you can use the ``any`` method on a Boolean DataFrame:

In [59]:
data[(data.abs()>3).any(axis="columns")]

,0,1,2,3
103,-0.335627,3.190082,0.518976,1.263475
188,0.660416,-2.095762,-3.458135,-0.043248
264,-1.337781,3.013570,1.578387,0.420259
310,3.173361,-1.361739,0.776512,-0.255989
337,-1.512399,3.309421,0.216194,2.051461
429,3.318278,0.063414,2.573696,-1.517655
433,0.337880,-0.139501,-3.048507,1.441885
528,1.001482,0.927736,-0.927412,-3.088709
639,-1.334268,1.748759,3.387302,-2.155885
797,3.213229,0.812480,-1.795666,-1.397913


The parentheses around ``data.abs() > 3`` are necessary in order to call the ``any`` method on the result of the comparison operation. 

Values can be set based on these criteria. Here is code to cap values outside the interval -3 to 3:

In [60]:
data[data.abs() > 3] = np.sign(data) * 3

data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,-0.000485,0.000564,-0.026656,-0.018038
std,1.017605,0.972711,0.988456,1.031485
min,-2.842948,-3.000000,-3.000000,-3.000000
25%,-0.748958,-0.652075,-0.674084,-0.712238
50%,0.026256,-0.053696,-0.050211,-0.000837
75%,0.673777,0.609422,0.651785,0.740628
max,3.000000,3.000000,3.000000,2.907953


The statement ``np.sign(data)`` produces 1 to -1 values based on whether the values in ``data`` are positive or negative:

In [62]:
np.sign(data).head()

,0,1,2,3
0,1.0,1.0,1.0,1.0
1,-1.0,1.0,-1.0,1.0
2,1.0,1.0,-1.0,-1.0
3,1.0,1.0,-1.0,-1.0
4,1.0,1.0,-1.0,-1.0


## Permutation and Random Sampling

Permuting (randomly reordering) a Series or the rows in a DataFrame is possible using the ``numpy.random.permutation`` function. Calling ``permutation`` with the length of the axis you want to permute produces an array of integers indicating the new ordering:

In [64]:
df = pd.DataFrame(np.arange(5 * 7).reshape((5, 7)))

df

,0,1,2,3,4,5,6
0,0,1,2,3,4,5,6
1,7,8,9,10,11,12,13
2,14,15,16,17,18,19,20
3,21,22,23,24,25,26,27
4,28,29,30,31,32,33,34
